# 02-Feature engineering causal et séparation anti-leakage

Ce notebook montre d'abord **manuellement** la création des partitions et de quelques features temporelles. Ensuite, la logique est appliquée de manière industrielle avec **src/features/build_features.py**.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
# Résout la racine du projet quel que soit le répertoire depuis lequel Jupyter a été lancé
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    # Si Jupyter est lancé depuis notebooks/, on remonte d'un niveau
    ROOT = ROOT.parent

DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
MODELS_DIR = ROOT / "models"

print("Project root :", ROOT)
print("Raw data     :", DATA_RAW)


Project root : C:\Users\Admin\Desktop\predictmaint-ai
Raw data     : C:\Users\Admin\Desktop\predictmaint-ai\data\raw


### Analyse du résultat

Les entrées brutes et les sorties transformées sont localisées explicitement. Cela rend l'enchaînement des notebooks traçable et évite de dépendre du répertoire depuis lequel Jupyter a été lancé.


## 1. Rechargement manuel du train brut


In [3]:
# Rechargement du train brut et calcul de la cible de maintenance prédictive (RUL + fenêtre de panne)
columns = [
    "engine_id", "cycle", "setting_1", "setting_2", "setting_3",
    *[f"sensor_{i}" for i in range(1, 22)],
]

train = pd.read_csv(
    DATA_RAW / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=columns,
)

# Horizon de prédiction : une panne est considérée imminente si elle survient dans les 30 prochains cycles
FAILURE_WINDOW = 30
# RUL (Remaining Useful Life) = nombre de cycles restants avant la panne du moteur
# (dernier cycle observé pour ce moteur - cycle courant)
train["RUL"] = train.groupby("engine_id")["cycle"].transform("max") - train["cycle"]
# Cible binaire : 1 si la panne est attendue dans la fenêtre définie, 0 sinon
train["failure_within_30_cycles"] = (train["RUL"] <= FAILURE_WINDOW).astype("int8")

print(train.shape)
display(train.head())


(20631, 28)


,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL,failure_within_30_cycles
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191,0
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190,0
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189,0
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188,0
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187,0


### Analyse du résultat

Les 20 631 lignes possèdent maintenant 28 colonnes : les 26 variables brutes, le RUL et la cible binaire. Les premières lignes, éloignées de la panne, sont logiquement étiquetées 0.


## 2. Split manuel par moteur avant transformation


In [4]:
# Split par moteur (et non par ligne), effectué AVANT toute transformation pour éviter toute fuite entre partitions
RANDOM_STATE = 42
engines = np.array(sorted(train["engine_id"].unique()))
rng = np.random.default_rng(RANDOM_STATE)
rng.shuffle(engines)  # mélange aléatoire reproductible des identifiants moteurs

# Répartition 70/15/15 moteurs (pas de lignes) entre train, calibration et validation
train_ids = set(engines[:70])
calibration_ids = set(engines[70:85])
validation_ids = set(engines[85:])

train_raw = train[train.engine_id.isin(train_ids)].copy().sort_values(["engine_id", "cycle"])
calibration_raw = train[train.engine_id.isin(calibration_ids)].copy().sort_values(["engine_id", "cycle"])
validation_raw = train[train.engine_id.isin(validation_ids)].copy().sort_values(["engine_id", "cycle"])

# Vérifie qu'aucun moteur n'apparaît dans plusieurs partitions (anti-leakage)
assert train_ids.isdisjoint(calibration_ids)
assert train_ids.isdisjoint(validation_ids)
assert calibration_ids.isdisjoint(validation_ids)

print("TRAIN       :", train_raw.engine_id.nunique(), len(train_raw))
print("CALIBRATION :", calibration_raw.engine_id.nunique(), len(calibration_raw))
print("VALIDATION  :", validation_raw.engine_id.nunique(), len(validation_raw))


TRAIN       : 70 14407
CALIBRATION : 15 3160
VALIDATION  : 15 3064


### Analyse du résultat

Les tailles 70/15/15 moteurs reproduisent exactement le split de l'EDA. Fixer la graine et sauvegarder le manifeste garantissent que toutes les étapes aval utilisent les mêmes groupes.


## 3. Exemple manuel de features causales

Pour une ligne au cycle **t**, seules les observations du **même moteur** avec un cycle **<= t** peuvent être utilisées.

Exemples :
- **lag_1** : valeur au cycle précédent ;
- **diff_1** : variation depuis le cycle précédent ;
- **mean_5** : moyenne des 5 derniers cycles ;
- **std_5** : volatilité récente.


In [ ]:
example = train_raw[["engine_id", "cycle", "sensor_11"]].copy()
g = example.groupby("engine_id", sort=False)["sensor_11"]

example["sensor_11_lag_1"] = g.shift(1)
example["sensor_11_diff_1"] = g.diff(1)
# reset_index + sort_index : nécessaire pour réaligner le résultat du rolling groupby
# (qui renvoie un MultiIndex engine_id/cycle) sur l'index original de example
example["sensor_11_mean_5"] = (
    g.rolling(window=5, min_periods=1)
     .mean()
     .reset_index(level=0, drop=True)
     .sort_index()
)
# min_periods=2 pour l'écart-type : il faut au moins 2 valeurs pour qu'il soit défini
example["sensor_11_std_5"] = (
    g.rolling(window=5, min_periods=2)
     .std()
     .reset_index(level=0, drop=True)
     .sort_index()
)

display(example.head(12))


,engine_id,cycle,sensor_11,sensor_11_lag_1,sensor_11_diff_1,sensor_11_mean_5,sensor_11_std_5
0,1,1,47.47,NaN,NaN,47.470,NaN
1,1,2,47.49,47.47,0.02,47.480,0.014142
2,1,3,47.27,47.49,-0.22,47.410,0.121655
3,1,4,47.13,47.27,-0.14,47.340,0.171659
4,1,5,47.28,47.13,0.15,47.328,0.151063
5,1,6,47.16,47.28,-0.12,47.266,0.141527
6,1,7,47.36,47.16,0.20,47.240,0.094074
7,1,8,47.24,47.36,-0.12,47.234,0.092628
8,1,9,47.29,47.24,0.05,47.266,0.073348
9,1,10,47.03,47.29,-0.26,47.216,0.127004


### Analyse du résultat

Le `lag_1` et la différence sont absents au premier cycle, ce qui est attendu faute de passé. Les moyennes et écarts glissants n'utilisent que les cycles déjà observés ; les valeurs initiales manquantes devront être imputées dans le pipeline du modèle.


## 4. Test manuel de causalité : modifier le futur ne doit pas changer le passé


In [ ]:
# Test unitaire manuel : une modification d'une valeur future ne doit pas changer les features calculées sur le passé
engine_id = int(train_raw["engine_id"].iloc[0])
g0 = train_raw[train_raw.engine_id == engine_id].head(25).copy()

# Référence : mean_5 calculée sur les données originales
base = g0[["engine_id", "cycle", "sensor_11"]].copy()
base["mean_5"] = (
    base.groupby("engine_id")["sensor_11"]
        .rolling(5, min_periods=1).mean()
        .reset_index(level=0, drop=True)
)

# On modifie artificiellement la dernière valeur (la plus future) du groupe
mutated = g0.copy()
mutated.loc[mutated.index[-1], "sensor_11"] += 9999
check = mutated[["engine_id", "cycle", "sensor_11"]].copy()
check["mean_5"] = (
    check.groupby("engine_id")["sensor_11"]
         .rolling(5, min_periods=1).mean()
         .reset_index(level=0, drop=True)
)

# La feature au 10e cycle (index 9) doit rester identique avant/après, car elle ne dépend que du passé
assert np.isclose(base.iloc[9]["mean_5"], check.iloc[9]["mean_5"])
print("OK : une valeur future ne modifie pas une feature passée.")


OK : une valeur future ne modifie pas une feature passée.


### Analyse du résultat

Le test passe : modifier une mesure future ne change aucune feature antérieure. C'est la propriété essentielle pour éviter une fuite temporelle lors de l'entraînement et en production.


## 5. Application industrialisée à chaque partition séparément

La fonction de production construit les mêmes familles de features, moteur par moteur. On l'appelle **séparément** sur TRAIN, CALIBRATION et VALIDATION.


In [7]:
import sys
# Ajout de la racine du projet au PYTHONPATH pour pouvoir importer le module src/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.features.build_features import build_causal_features

train_features = build_causal_features(train_raw)
calibration_features = build_causal_features(calibration_raw)
validation_features = build_causal_features(validation_raw)

print("TRAIN       :", train_features.shape)
print("CALIBRATION :", calibration_features.shape)
print("VALIDATION  :", validation_features.shape)


TRAIN       : (14407, 250)
CALIBRATION : (3160, 250)
VALIDATION  : (3064, 250)


### Analyse du résultat

Les trois partitions conservent leur nombre de lignes et aboutissent au même schéma de 250 colonnes. La transformation est donc structurellement cohérente sans mélanger leurs historiques.


In [8]:
# Colonnes ajoutées par le feature engineering (celles présentes dans train_features mais pas dans train_raw)
engineered = [c for c in train_features.columns if c not in train_raw.columns]
print("Nombre de features dérivées :", len(engineered))
print(engineered[:50])


Nombre de features dérivées : 222
['sensor_2_lag_1', 'sensor_2_lag_3', 'sensor_2_diff_1', 'sensor_2_mean_5', 'sensor_2_std_5', 'sensor_2_min_5', 'sensor_2_max_5', 'sensor_2_range_5', 'sensor_2_dev_mean_5', 'sensor_2_mean_10', 'sensor_2_std_10', 'sensor_2_min_10', 'sensor_2_max_10', 'sensor_2_range_10', 'sensor_2_dev_mean_10', 'sensor_2_mean_20', 'sensor_2_std_20', 'sensor_2_min_20', 'sensor_2_max_20', 'sensor_2_range_20', 'sensor_2_dev_mean_20', 'sensor_2_ewm_10', 'sensor_3_lag_1', 'sensor_3_lag_3', 'sensor_3_diff_1', 'sensor_3_mean_5', 'sensor_3_std_5', 'sensor_3_min_5', 'sensor_3_max_5', 'sensor_3_range_5', 'sensor_3_dev_mean_5', 'sensor_3_mean_10', 'sensor_3_std_10', 'sensor_3_min_10', 'sensor_3_max_10', 'sensor_3_range_10', 'sensor_3_dev_mean_10', 'sensor_3_mean_20', 'sensor_3_std_20', 'sensor_3_min_20', 'sensor_3_max_20', 'sensor_3_range_20', 'sensor_3_dev_mean_20', 'sensor_3_ewm_10', 'sensor_4_lag_1', 'sensor_4_lag_3', 'sensor_4_diff_1', 'sensor_4_mean_5', 'sensor_4_std_5', 'sens

### Analyse du résultat

Les 222 variables dérivées décrivent retards, différences, fenêtres glissantes et lissage exponentiel. Cette richesse augmente fortement la dimension (250 colonnes au total), ce qui rend l'étape de sélection suivante indispensable.


## 6. Variables interdites au modèle

`engine_id`, `RUL` et la cible peuvent être conservés dans les tables de travail pour la traçabilité, mais ils doivent être retirés de `X` au moment de l'apprentissage.


In [9]:
forbidden = {"engine_id", "RUL", "failure_within_30_cycles", "max_cycle"}
print("Variables de vérité terrain / identité à exclure de X :", sorted(forbidden))


Variables de vérité terrain / identité à exclure de X : ['RUL', 'engine_id', 'failure_within_30_cycles', 'max_cycle']


### Analyse du résultat

`engine_id`, le RUL, la cible et le cycle terminal servent à la traçabilité ou à construire la vérité terrain, mais ne doivent jamais entrer dans `X`. Leur exclusion empêche une fuite directe de l'identité ou du futur.


## Conclusion

Le feature engineering est maintenant :

1. séparé par moteur ;
2. causal dans le temps ;
3. appliqué après split ;
4. identique entre expérimentation et production grâce au module `src/features/`.
